# Sovereign Byte Firewall — Kaggle GPU Training (v2 masking)

Retrain with the fixed masking scheme (TLS continuation, stochastic header fields, QUIC).

**Rules baked into this notebook — do not undo them:**
1. **Train on the MONDAY file only** (benign-only day). Wednesday is an ATTACK day: training on it contaminates the model AND it is the held-out evaluation day for `evaluate_cic_days.py`.
2. **1 epoch.** Repeating epochs over the same file caused real regression (32% -> ~6% held-out detection).
3. **Plain CrossEntropy.** FocalLoss proven worse.
4. **Fresh start.** Old checkpoints were trained under the old masking — not resumable. Checkpoints push to a NEW HF repo to avoid gs-number collisions.

Make sure GPU accelerator is ON: Settings > Accelerator > GPU T4 x2 or P100.

In [ ]:
import subprocess, sys
for pkg in ['scapy', 'huggingface_hub>=0.25', 'accelerate']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
print('Dependencies installed')

In [ ]:
import os, subprocess
REPO_DIR = '/kaggle/working/sovereign-byte-firewall'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/TheSPST/sovereign-byte-firewall.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
# Sanity: this run REQUIRES the v2 masking commits
log = subprocess.run(['git', 'log', '--oneline', '-8'], capture_output=True, text=True).stdout
print(log)
assert 'stochastic header fields' in subprocess.run(['git', 'log', '--format=%s', '-30'], capture_output=True, text=True).stdout, \
    'v2 masking commits missing — did you push c59df36+ to GitHub?'
print('Repo ready (v2 masking confirmed) at:', os.getcwd())

In [ ]:
import os
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in via Kaggle Secret')
except Exception:
    HF_TOKEN = ''  # paste token here as fallback
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('Logged in via token')
    else:
        print('WARNING: No HF token - checkpoints will NOT be uploaded')
# NEW repo: v2-masking checkpoints must not mix with old-masking gs-numbers
os.environ['HF_REPO_ID'] = 'spst01/sovereign-byte-firewall-v2-masking'
print('Checkpoint backup repo:', os.environ['HF_REPO_ID'])

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

# TRAIN ON MONDAY (benign-only) — NEVER Wednesday (attack day + eval day).
DATASET_PATH = None

# 1) Manually-attached Kaggle datasets first
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if 'onday' in f and (f.endswith('.pcap') or f.endswith('.gz')):
            DATASET_PATH = os.path.join(root, f)
            print('Monday dataset found (Kaggle input):', DATASET_PATH)
            break
    if DATASET_PATH:
        break

# 2) Otherwise pull from the HF dataset repo
if not DATASET_PATH:
    repo = 'spst01/sovereign-byte-dataset'
    files = HfApi().list_repo_files(repo, repo_type='dataset')
    monday = [f for f in files if 'onday' in f and (f.endswith('.pcap') or f.endswith('.gz'))]
    print('Repo files:', files)
    assert monday, ('No Monday file in ' + repo + ' — upload it or attach as Kaggle dataset. '
                    'Do NOT fall back to Wednesdaytruncated.gz (attack day = contaminated training + eval day).')
    DATASET_PATH = hf_hub_download(repo_id=repo, filename=monday[0], repo_type='dataset',
                                   local_dir='/kaggle/working/')
    print('Monday dataset downloaded:', DATASET_PATH)

assert 'ednesday' not in os.path.basename(DATASET_PATH), 'Refusing to train on Wednesday (attack/eval day)!'
print(f'Size: {os.path.getsize(DATASET_PATH)/1e9:.2f} GB')

In [ ]:
import torch, subprocess
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected! Turn on GPU accelerator in Settings.')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU[{i}]: {name} - {vram:.1f}GB VRAM')
subprocess.run(['nvidia-smi'])

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/sovereign-byte-firewall')
os.chdir('/kaggle/working/sovereign-byte-firewall')

sys.argv = [
    'run_kaggle.py',
    '--dataset_path', DATASET_PATH,
    '--epochs', '1',                 # >1 epoch = proven regression
    '--batch_size', '32',
    '--max_sequence_length', '512',
    '--lr', '5e-4',
    '--use_focal_loss', 'false',     # FocalLoss proven worse
    '--checkpoints_dir', '/kaggle/working/checkpoints',
    # '--total_steps', '1000000',    # uncomment with the known windows/epoch for an exact LR schedule
]

from run_kaggle import main
main()

In [ ]:
import glob, os
ckpts = glob.glob('/kaggle/working/checkpoints/*.pt')
print(f'{len(ckpts)} checkpoint(s) saved:')
for c in sorted(ckpts):
    print(f'  {os.path.basename(c)} ({os.path.getsize(c)/1e6:.1f} MB)')
print('View on HF Hub: https://huggingface.co/spst01/sovereign-byte-firewall-v2-masking')
print('Monitor from your Mac: python scripts/aikosh_monitor.py --hf_repo spst01/sovereign-byte-firewall-v2-masking')